In [9]:
import numpy as np
import librosa
import tensorflow as tf

print("Yapay zeka modeli uyanıyor...")
model = tf.keras.models.load_model('multi_instrument_model.keras')

enstruman_sozlugu = {
    'cel': 'Çello (Cello)',
    'cla': 'Klarnet (Clarinet)',
    'flu': 'Flüt (Flute)',
    'gac': 'Akustik Gitar (Acoustic Guitar)',
    'gel': 'Elektro Gitar (Electric Guitar)',
    'org': 'Org (Organ)',
    'pia': 'Piyano (Piano)',
    'sax': 'Saksafon (Saxophone)',
    'tru': 'Trompet (Trumpet)',
    'vio': 'Keman (Violin)',
    'voi': 'İnsan Sesi (Voice)'
}
sinif_kodlari = list(enstruman_sozlugu.keys())

def enstruman_tahmin_et(ses_yolu):
    print(f"\n '{ses_yolu}' analiz ediliyor...")
    
    try:
        y, sr = librosa.load(ses_yolu, sr=None, mono=True)
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        
        hedef_genislik = 130
        mevcut_genislik = mel_spec_db.shape[1]
        
        if mevcut_genislik < hedef_genislik: 
            pad_width = hedef_genislik - mevcut_genislik
            mel_spec_db = np.pad(mel_spec_db, pad_width=((0, 0), (0, pad_width)), mode='constant')
        else: 
            mel_spec_db = mel_spec_db[:, :hedef_genislik]
            
        mel_spec_db_norm = (mel_spec_db - np.min(mel_spec_db)) / (np.max(mel_spec_db) - np.min(mel_spec_db) + 1e-8)
        
        girdi = mel_spec_db_norm[np.newaxis, ..., np.newaxis]
        
        tahminler = model.predict(girdi, verbose=0)
        
        en_yuksek_ihtimal_indeksi = np.argmax(tahminler)
        guven_skoru = tahminler[0][en_yuksek_ihtimal_indeksi] * 100
        tahmin_edilen_kod = sinif_kodlari[en_yuksek_ihtimal_indeksi]
        
        print(f"Sonuç: {enstruman_sozlugu[tahmin_edilen_kod]} (Eminlik Oranı: %{guven_skoru:.2f})")
        
    except Exception as e:
        print(f"Hata oluştu: {e}")

test_dosyasi = "IRMAS-TrainingData/tru/[tru][cla]1870__3.wav" 
enstruman_tahmin_et(test_dosyasi)

Yapay zeka modeli uyanıyor...

🎧 'IRMAS-TrainingData/tru/[tru][cla]1870__3.wav' analiz ediliyor...
✅ Sonuç: Trompet (Trumpet) (Eminlik Oranı: %49.72)
